In [ ]:
import pandas as pd
from datasets import load_dataset

xsum = pd.read_csv('Model_generated_content/XSum/gpt4_vs_Xsum_1~1000.csv')
brio = pd.read_json('Model_generated_content/CNNDM/pegasus_generated_predictions_twice.json')

data = pd.DataFrame({
    'original_article': xsum['original_article'][:1000],
    'Summary_1': brio['summary'][:1000],
})

data.to_csv('Model_generated_content/XSum/Original_brio_vs_Xsum_1~1000.csv', index=False)

In [ ]:
import pandas as pd

dataset = load_dataset('cnn_dailymail', '3.0.0', split='test')
dataset = pd.DataFrame(dataset)
target = pd.read_json('Model_generated_content/CNNDM/pegasus_generated_predictions_twice.json')
target2 = pd.read_json('Model_generated_content/CNNDM/Original_pegasus_generated_predictions.json')

dataset.drop_duplicates(subset=['article'], inplace=True, ignore_index=True)
target.drop_duplicates(subset=['document'], inplace=True, ignore_index=True)
target2.drop_duplicates(subset=['document'], inplace=True, ignore_index=True)

dataset.replace('\n', ' ', regex=True, inplace=True)
target.replace('\n', ' ', regex=True, inplace=True)
target2.replace('\n', ' ', regex=True, inplace=True)

correct = pd.DataFrame()
correct2 = pd.DataFrame()

for index in range(len(dataset)):
    correct = pd.concat([correct, target.loc[target['document'] == dataset['article'][index]]], axis=0, ignore_index=True)

for index in range(len(dataset)):
    correct2 = pd.concat([correct2, target2.loc[target2['document'] == dataset['article'][index]]], axis=0, ignore_index=True)

cnndm = pd.DataFrame({
    'original_article': dataset['article'],
    'Summary_1': correct2['summary'],
    'Summary_2': correct['summary'],
})

print(len(cnndm))
cnndm.to_csv('Model_generated_content/CNNDM/pegasus_vs_Original_pegasus_Full_v2.csv', index=False)

In [ ]:
import pandas as pd
from datasets import load_dataset


xsum = pd.read_json('datasets/xsum/sorted_test_gpt4_turbo.json')
xsum_ref = load_dataset('xsum', split='test')
xsum_ref = pd.DataFrame(xsum_ref)
brio = pd.read_json('Model_generated_content/XSum/brio_generated_predictions.json')
bart = pd.read_json('Model_generated_content/XSum/bart_generated_predictions.json')
pegasus = pd.read_json('Model_generated_content/XSum/pegasus_generated_predictions.json')

# Summary_1 = gpt4, Summary_2 = brio, Summary_3 = bart, Summary_4 = pegasus, Summary_5 = xsum_ref
df = pd.DataFrame({
    'original_article': xsum['document'],
    'Summary_1': xsum['summary'],
    'Summary_2': brio['summary'],
    'Summary_3': bart['summary'],
    'Summary_4': pegasus['summary'],
    'Summary_5': xsum_ref['summary'],

})

df.replace('\n', ' ', regex=True, inplace=True)

df.to_csv('Model_generated_content/XSum/All_models_ranking_Full.csv', index=False)